# Transfer Learning & Fine-Tuning

## 1. Big Picture

Transfer learning starts from a simple observation:

> A model that has already learned useful representations from a large dataset should not need to relearn everything from scratch for every new task.

```mermaid
flowchart TD
    A["Pre-training"] --> B["Pretrained Model"]
    B --> C["Transfer Learning"]

    C --> D["Feature Extraction / Linear Probing"]
    D --> D1["Freeze Backbone<br/>Train New Head"]

    C --> E["Fine-Tuning"]
    E --> E1["Partial Fine-Tuning"]
    E --> E2["Full Fine-Tuning"]
    E --> E3["Parameter-Efficient Fine-Tuning<br/>(PEFT)"]

    E3 --> F1["Adapters"]
    E3 --> F2["LoRA"]
    E3 --> F3["Prefix Tuning"]
    E3 --> F4["Prompt Tuning"]

    F2 --> G1["QLoRA"]
    F2 --> G2["LoRA-XS"]
    F2 --> G3["TinyLoRA"]
```

The entire idea can be summarized as:

$$
\boxed{\text{Pre-training learns reusable knowledge}}
$$

$$
\boxed{\text{Transfer Learning reuses that knowledge}}
$$

$$
\boxed{\text{Fine-Tuning adapts that knowledge to a target task}}
$$

---

## 2. Why Transfer Learning?

Suppose we want to train a CNN for a new image classification task.

Training from scratch means starting from random parameters:

$$
\theta_{\text{random}}
$$

and forcing the model to learn the entire representation hierarchy:

```text
Pixels
  ↓
Edges
  ↓
Textures
  ↓
Shapes
  ↓
Object Parts
  ↓
High-level Concepts
  ↓
Target Prediction
```

However, models such as ResNet or EfficientNet pretrained on a large dataset have already learned many useful visual representations.

Instead of starting from:

$$
\theta_{\text{random}}
$$

we start from:

$$
\theta_{\text{pretrained}}
$$

and reuse the knowledge learned during pre-training.

This is the fundamental idea of **Transfer Learning**.

---

## 3. What Is Actually Being Transferred?

The most important thing being transferred is usually not the original classifier.

It is the learned **representation**.

A CNN generally learns hierarchical representations:

```text
Early Layers
    ↓
Edges, colors, simple textures
    ↓
Middle Layers
    ↓
Patterns and object parts
    ↓
Later Layers
    ↓
High-level semantic features
```

In general:

$$
\boxed{\text{Early layers tend to learn more generic features}}
$$

while:

$$
\boxed{\text{Later layers tend to become more task-specific}}
$$

This is why the lower layers of a pretrained model can often be reused across many downstream tasks.

---

## 4. Mathematical View

A pretrained model can be decomposed as:

$$
f(x)=g_{\psi}(h_{\phi}(x))
$$

where:

- $h_{\phi}$: backbone / feature extractor
- $g_{\psi}$: prediction head

The backbone transforms the input into a representation:

$$
z=h_{\phi}(x)
$$

and the prediction head maps that representation to the final prediction:

$$
\hat{y}=g_{\psi}(z)
$$

After pre-training, we obtain:

$$
\phi_0,\psi_0
$$

For a new target task, the original prediction head is often replaced, while the pretrained backbone can be reused.

---

# Feature Extraction

## 5. Feature Extraction

The simplest form of transfer learning is **Feature Extraction**.

The pretrained backbone is frozen:

$$
\phi=\phi_0
$$

and only a new task-specific head is trained:

$$
\psi_{\text{target}}
$$

The architecture becomes:

```text
Input
  ↓
Pretrained Backbone
  ↓
Frozen Representation
  ↓
New Task-Specific Head
  ↓
Prediction
```

During training:

```text
Backbone       → Frozen
New Head       → Trainable
```

The optimization problem becomes:

$$
\min_{\psi}
L\left(
g_{\psi}(h_{\phi_0}(x)),y
\right)
$$

The backbone does not change.

---

## 6. Why Feature Extraction Works

The pretrained backbone already provides:

$$
x\rightarrow z
$$

where $z$ is a useful representation.

Therefore, instead of learning:

$$
\text{Input}
\rightarrow
\text{Representation}
\rightarrow
\text{Prediction}
$$

from scratch, the target task only needs to learn:

$$
\boxed{
\text{Pretrained Representation}
\rightarrow
\text{Target Prediction}
}
$$

This reduces the number of trainable parameters and is especially useful when the target dataset is small.

---

## 7. Linear Probing

**Linear Probing** is a special case of feature extraction.

The backbone is completely frozen and only a linear classifier is trained:

$$
\hat{y}=Wz+b
$$

where:

$$
z=h_{\phi_0}(x)
$$

Therefore:

$$
\boxed{
\text{Linear Probe}
=
\text{Frozen Representation}
+
\text{Linear Classifier}
}
$$

Linear probing is also useful for evaluating the quality of a learned representation.

If a simple linear classifier performs well, the pretrained representation already separates the target concepts effectively.

---

## 8. Limitation of Feature Extraction

Feature extraction assumes that the pretrained representation is already suitable for the target task.

Consider:

```text
Source Domain:
Natural Images

        ↓

Target Domain:
Medical X-rays
```

The optimal target representation may differ significantly from the pretrained one.

If the backbone is frozen:

$$
\phi=\phi_0
$$

then the representation cannot adapt.

This motivates **Fine-Tuning**.

---

# Fine-Tuning

## 9. Fine-Tuning

Fine-tuning starts from pretrained parameters and continues training them on the target dataset.

Instead of:

$$
\theta_{\text{random}}
\rightarrow
\theta_{\text{target}}
$$

we perform:

$$
\boxed{
\theta_{\text{pretrained}}
\rightarrow
\theta_{\text{target}}
}
$$

using the target-task loss:

$$
\theta
\leftarrow
\theta-\eta\nabla_{\theta}L_{\text{target}}
$$

Conceptually:

```text
Useful Pretrained Representation
            ↓
Controlled Adaptation
            ↓
Target-Specific Representation
```

Therefore:

$$
\boxed{
\text{Feature Extraction}
=
\text{Reuse Representation}
}
$$

while:

$$
\boxed{
\text{Fine-Tuning}
=
\text{Reuse + Adapt Representation}
}
$$

---

## 10. Partial Fine-Tuning

Fine-tuning does not require updating the entire model.

A common strategy is to freeze the earlier layers and train only the later layers.

```text
Input
  ↓
Block 1       Frozen
  ↓
Block 2       Frozen
  ↓
Block 3       Frozen
  ↓
Block 4       Trainable
  ↓
Block 5       Trainable
  ↓
New Head      Trainable
```

The intuition comes from the hierarchical nature of learned features:

```text
Early Layers
→ more generic features
→ preserve them

Later Layers
→ more task-specific features
→ adapt them
```

Partial fine-tuning provides a middle ground between feature extraction and full fine-tuning.

---

## 11. Transfer Learning as a Spectrum

Transfer learning is better understood as a continuum rather than a binary choice.

```text
Less Adaptation
      │
      ▼

Feature Extraction
      │
      ▼
Unfreeze Last Layer
      │
      ▼
Unfreeze Last Few Blocks
      │
      ▼
Partial Fine-Tuning
      │
      ▼
Full Fine-Tuning

      ▲
      │
More Adaptation
```

The number of layers to unfreeze is a design decision.

There is no universal rule such as:

> Always freeze a fixed percentage of the network.

---

## 12. Full Fine-Tuning

In **Full Fine-Tuning**, every pretrained parameter becomes trainable.

If:

$$
\theta_0
$$

denotes the pretrained parameters, then:

$$
\boxed{
\theta_{\text{target}}
=
\theta_0+\Delta\theta
}
$$

where all components of $\Delta\theta$ may be updated.

For a Transformer:

```text
Embeddings        Trainable
Attention         Trainable
Feed Forward      Trainable
LayerNorm         Trainable
...
Output Head       Trainable
```

Full fine-tuning provides the largest adaptation capacity because the optimizer can modify the entire network.

However, it also requires:

- more GPU memory,
- gradients for all trainable parameters,
- optimizer states for all trainable parameters,
- larger task-specific checkpoints,
- sufficient target data to avoid excessive overfitting.

---

## 13. How Much Should We Fine-Tune?

Two important factors determine how much adaptation is appropriate.

### 13.1 Target Dataset Size

When the target dataset is small:

$$
N_{\text{target}}\downarrow
$$

training too many parameters may lead to overfitting.

Therefore, freezing more of the model is often preferable.

When the target dataset is larger:

$$
N_{\text{target}}\uparrow
$$

more aggressive fine-tuning becomes possible.

---

### 13.2 Source–Target Similarity

If the source and target domains are similar:

```text
ImageNet
   ↓
Dogs vs Cats
```

the pretrained representation may already be highly useful.

If they are very different:

```text
Natural Images
   ↓
Medical Images
```

stronger adaptation may be required.

A useful heuristic is:

| Target Data | Source–Target Similarity | Initial Strategy |
|---|---|---|
| Small | High | Feature Extraction |
| Small | Moderate | Partial Fine-Tuning |
| Large | High | Partial / Full Fine-Tuning |
| Large | Low | Stronger / Full Fine-Tuning |

These are guidelines rather than strict rules.

---

## 14. Why Fine-Tuning Usually Uses a Smaller Learning Rate

Pretrained parameters already encode useful knowledge.

We do not want:

```text
Good Pretrained Representation
            ↓
Very Large Updates
            ↓
Destroy Useful Knowledge
```

Instead:

```text
Good Pretrained Representation
            ↓
Small Controlled Updates
            ↓
Target-Specific Representation
```

Therefore, fine-tuning commonly uses a smaller learning rate than training from scratch.

The intuition is:

$$
\boxed{
\theta_{\text{target}}
\approx
\theta_{\text{pretrained}}
+
\text{Useful Adaptation}
}
$$

rather than learning an entirely new representation.

---

# Transfer Learning in Computer Vision

## 15. Typical Computer Vision Workflow

Suppose we have a ResNet-50 pretrained on ImageNet.

### Step 1 — Replace the Original Head

```text
Pretrained ResNet-50
        ↓
Remove ImageNet Head
        ↓
Add Target Head
```

### Step 2 — Feature Extraction

```text
Backbone     Frozen
Head         Trainable
```

Train the new classifier first.

### Step 3 — Partial Fine-Tuning

```text
Early Blocks    Frozen
Late Blocks     Trainable
Head            Trainable
```

Use a relatively small learning rate.

### Step 4 — Full Fine-Tuning

If sufficient data and compute are available:

```text
Entire Backbone   Trainable
Head              Trainable
```

A useful experiment is therefore:

```text
1. Pretrained Model + New Head
2. Frozen Backbone
3. Unfreeze Several Final Blocks
4. Full Fine-Tuning
```

These settings can be compared using:

- validation accuracy,
- training time,
- trainable parameter count,
- generalization,
- overfitting behavior.

---

# Parameter-Efficient Fine-Tuning

## 16. Why Full Fine-Tuning Becomes Difficult for LLMs

For a CNN with tens of millions of parameters, full fine-tuning may still be practical.

For modern language models:

$$
7B,\ 14B,\ 70B,\ldots
$$

the situation changes dramatically.

Training memory includes more than model weights:

```text
Weights
+
Gradients
+
Optimizer States
+
Activations
```

Therefore, updating billions of parameters can become extremely expensive.

This leads to an important question:

> Do we really need to update every parameter of a pretrained model?

This motivates:

$$
\boxed{\text{Parameter-Efficient Fine-Tuning}}
$$

---

## 17. Parameter-Efficient Fine-Tuning — PEFT

PEFT freezes most or all pretrained parameters and introduces only a small set of trainable parameters.

Let:

$$
\theta_0
$$

be the pretrained parameters.

PEFT keeps:

$$
\boxed{\theta_0\text{ frozen}}
$$

and introduces trainable parameters:

$$
\phi
$$

where:

$$
\boxed{
|\phi|\ll|\theta_0|
}
$$

The model can be written as:

$$
f(x;\theta_0,\phi)
$$

PEFT is not one specific technique.

It is a family of methods:

```text
PEFT
├── Adapters
├── LoRA
│    ├── QLoRA
│    ├── LoRA-XS
│    └── TinyLoRA
├── Prefix Tuning
└── Prompt Tuning
```

---

# Adapters

## 18. Adapters

Adapters insert small trainable neural modules into a frozen pretrained network.

Suppose:

$$
h\in\mathbb{R}^{d}
$$

is a hidden representation.

An adapter commonly uses a bottleneck:

$$
d\rightarrow r\rightarrow d
$$

where:

$$
r\ll d
$$

A simplified adapter can be written as:

$$
\boxed{
h'
=
h+
W_{\text{up}}
\sigma(W_{\text{down}}h)
}
$$

The pretrained model remains frozen.

Only:

$$
W_{\text{down}},W_{\text{up}}
$$

are trained.

Conceptually:

```text
Original Representation
        ↓
Small Trainable Module
        ↓
Task-Specific Correction
        ↓
Updated Representation
```

Therefore:

$$
\boxed{
\text{Adapter}
=
\text{Small Trainable Residual Module}
}
$$

The adapter does not replace the pretrained knowledge.

It adds a small task-specific correction on top of it.

---

# LoRA

## 19. LoRA — Low-Rank Adaptation

LoRA is one of the most important PEFT techniques.

Consider a pretrained linear transformation:

$$
y=W_0x
$$

Full fine-tuning would learn:

$$
W'=W_0+\Delta W
$$

where:

$$
\Delta W
$$

has the same shape as $W_0$.

If:

$$
W_0\in\mathbb{R}^{d_{\text{out}}\times d_{\text{in}}}
$$

then a full weight update requires:

$$
d_{\text{out}}d_{\text{in}}
$$

trainable parameters.

LoRA instead assumes that the task-specific update can be represented using a low-rank factorization:

$$
\boxed{
\Delta W=BA
}
$$

where:

$$
A\in\mathbb{R}^{r\times d_{\text{in}}}
$$

and:

$$
B\in\mathbb{R}^{d_{\text{out}}\times r}
$$

with:

$$
r\ll \min(d_{\text{in}},d_{\text{out}})
$$

The forward pass becomes:

$$
\boxed{
y=W_0x+BAx
}
$$

where:

$$
W_0\text{ is frozen}
$$

and only:

$$
A,B
$$

are trainable.

---

## 20. Why LoRA Saves Parameters

Suppose:

$$
W\in\mathbb{R}^{4096\times4096}
$$

A full update requires:

$$
4096^2
=
16,777,216
$$

trainable parameters.

Now choose:

$$
r=8
$$

LoRA uses:

$$
A\in\mathbb{R}^{8\times4096}
$$

and:

$$
B\in\mathbb{R}^{4096\times8}
$$

Therefore:

$$
N_{\text{LoRA}}
=
8(4096)+4096(8)
$$

$$
=
65,536
$$

Instead of approximately:

$$
16.8M
$$

parameters, only approximately:

$$
65K
$$

parameters are trained for that matrix.

The central idea is:

$$
\boxed{
\text{Do not learn a full weight update}
}
$$

Instead:

$$
\boxed{
\text{Learn a low-rank weight update}
}
$$

---

## 21. The Deeper Idea Behind LoRA

LoRA does **not** claim that the pretrained matrix:

$$
W_0
$$

is low-rank.

Instead, LoRA constrains only the task-specific change:

$$
\boxed{\Delta W}
$$

through:

$$
\boxed{
\Delta W=BA
}
$$

The pretrained model already contains a powerful set of representations and capabilities.

The downstream task may only require a relatively low-dimensional modification of those capabilities.

Therefore:

$$
\boxed{
\text{Pretrained Knowledge}
+
\text{Small Low-Rank Adaptation}
}
$$

can be sufficient for many downstream tasks.

---

# QLoRA

## 22. QLoRA

QLoRA extends LoRA by reducing the memory required to store the frozen base model.

With ordinary LoRA:

```text
Base Model
   ↓
Frozen

LoRA A, B
   ↓
Trainable
```

With QLoRA:

```text
Base Model
   ↓
4-bit Quantization
   ↓
Frozen

LoRA A, B
   ↓
Trainable
```

Therefore:

$$
\boxed{
\text{QLoRA}
=
\text{Quantized Frozen Base Model}
+
\text{LoRA}
}
$$

The important distinction is:

$$
\boxed{
\text{LoRA reduces trainable parameters}
}
$$

while:

$$
\boxed{
\text{QLoRA additionally reduces the memory used by the frozen base model}
}
$$

In other words:

```text
LoRA
→ Make adaptation cheap

QLoRA
→ Make adaptation cheap
  +
  Make the frozen backbone cheaper to store
```

---

# LoRA-XS and TinyLoRA

## 23. LoRA-XS

LoRA still trains two low-rank matrices:

$$
A,B
$$

LoRA-XS further restricts the adaptation space.

Suppose a pretrained weight matrix has a low-dimensional decomposition:

$$
W\approx U\Sigma V^\top
$$

Instead of learning two entirely new low-rank matrices, LoRA-XS can reuse directions derived from the pretrained weight and train a much smaller matrix:

$$
R
$$

Conceptually:

$$
\boxed{
\Delta W
=
U\Sigma R V^\top
}
$$

where the pretrained directions remain fixed and only $R$ is adapted.

The main idea is:

$$
\boxed{
\text{LoRA-XS compresses the LoRA adaptation space even further}
}
$$

The detailed implementation is less important than understanding this progression.

---

## 24. TinyLoRA

TinyLoRA pushes the same idea further.

The conceptual progression is:

$$
\boxed{
\text{Full Fine-Tuning}
\rightarrow
\text{LoRA}
\rightarrow
\text{LoRA-XS}
\rightarrow
\text{TinyLoRA}
}
$$

with progressively fewer trainable degrees of freedom.

Instead of directly learning a matrix:

$$
R
$$

TinyLoRA can represent it using fixed matrices:

$$
P_1,P_2,\ldots,P_k
$$

and train only a small set of coefficients:

$$
v_1,v_2,\ldots,v_k
$$

such that:

$$
\boxed{
R
=
\sum_i v_iP_i
}
$$

Conceptually:

```text
LoRA
→ Learn low-rank matrices

LoRA-XS
→ Learn a smaller adaptation matrix

TinyLoRA
→ Learn only tiny adaptation coefficients
```

The key idea is:

$$
\boxed{
\text{TinyLoRA aggressively reduces the number of trainable degrees of freedom}
}
$$

---

# Prompt Tuning

## 25. Prompt Tuning

Prompt Tuning follows a different strategy from LoRA.

Instead of modifying model weights, the pretrained model remains frozen.

A set of continuous trainable vectors is added to the input.

Suppose the original input embeddings are:

$$
x=[x_1,x_2,\ldots,x_n]
$$

Prompt Tuning learns:

$$
P=[p_1,p_2,\ldots,p_k]
$$

and feeds:

$$
\boxed{
x'=[P;x]
}
$$

to the model.

The vectors:

$$
p_1,p_2,\ldots,p_k
$$

are not necessarily human-readable words.

They are continuous embeddings optimized by gradient descent.

Conceptually:

```text
Trainable Soft Prompt
        +
Real Input Tokens
        ↓
Frozen Transformer
        ↓
Prediction
```

Therefore:

$$
\boxed{
\text{Prompt Tuning}
=
\text{Learn a Continuous Task-Specific Prompt}
}
$$

---

## 26. Prompt Engineering vs Prompt Tuning

These two ideas should not be confused.

### Prompt Engineering

A human manually writes a natural-language instruction:

```text
Classify the sentiment of the following sentence...
```

No gradient-based training is required.

### Prompt Tuning

The system learns continuous vectors:

$$
p_1,p_2,\ldots,p_k
$$

through gradient descent.

Therefore:

$$
\boxed{
\text{Prompt Engineering}
\neq
\text{Prompt Tuning}
}
$$

---

# Prefix Tuning

## 27. Prefix Tuning

Prefix Tuning also keeps the pretrained model frozen.

However, instead of only adding trainable vectors at the input level, it introduces trainable prefixes that influence the attention mechanism.

Conceptually, attention keys and values can be extended as:

$$
K'=[P^K;K]
$$

$$
V'=[P^V;V]
$$

The attention mechanism can therefore access:

```text
Real Token Information
        +
Learned Task-Specific Prefix
```

A useful mental model is:

$$
\boxed{
\text{Prefix Tuning}
=
\text{Learn Task-Specific Attention Memory}
}
$$

---

## 28. Prompt Tuning vs Prefix Tuning

### Prompt Tuning

```text
Soft Prompt
    ↓
Input Embeddings
    ↓
Transformer Layers
```

The learned information enters primarily through the input.

### Prefix Tuning

```text
Transformer Layer 1 ← Prefix
Transformer Layer 2 ← Prefix
Transformer Layer 3 ← Prefix
...
```

The learned prefix can directly influence attention across multiple layers.

Therefore:

$$
\boxed{
\text{Prompt Tuning}
=
\text{Learn What to Feed the Model}
}
$$

while:

$$
\boxed{
\text{Prefix Tuning}
=
\text{Learn Additional Memory for Attention}
}
$$

---

# PEFT Comparison

## 29. Comparison of PEFT Methods

| Method | Base Model | Trainable Component | Core Idea |
|---|---|---|---|
| **Adapters** | Frozen | Small neural modules | Add task-specific computation |
| **LoRA** | Frozen | Low-rank $A,B$ | Learn low-rank weight updates |
| **QLoRA** | Frozen + Quantized | LoRA $A,B$ | LoRA with a low-memory base model |
| **LoRA-XS** | Frozen | Smaller low-dimensional update | Compress LoRA further |
| **TinyLoRA** | Frozen | Tiny coefficients | Extreme adaptation compression |
| **Prompt Tuning** | Frozen | Soft embeddings | Learn continuous prompts |
| **Prefix Tuning** | Frozen | Attention prefixes | Learn task-specific attention memory |

Among these methods, the most important ones to understand deeply are:

$$
\boxed{\text{LoRA}}
$$

and:

$$
\boxed{\text{QLoRA}}
$$

LoRA-XS and TinyLoRA are mainly useful for understanding how parameter-efficient adaptation can be pushed toward increasingly smaller update spaces.

---

# Unified View

## 30. Fine-Tuning Through the Perspective of $\Delta\theta$

A powerful way to connect the entire topic is:

$$
\boxed{
\theta_{\text{target}}
=
\theta_{\text{pretrained}}
+
\Delta\theta
}
$$

The central question becomes:

> **How much freedom should $\Delta\theta$ have?**

### Feature Extraction

$$
\boxed{
\Delta\theta_{\text{backbone}}=0
}
$$

The backbone cannot change.

### Partial Fine-Tuning

Only selected parts of:

$$
\Delta\theta
$$

are allowed to change.

### Full Fine-Tuning

$$
\Delta\theta
$$

may modify the entire model.

### LoRA

Instead of learning an arbitrary matrix update:

$$
\Delta W
$$

we constrain it to:

$$
\boxed{
\Delta W=BA
}
$$

### LoRA-XS / TinyLoRA

The adaptation space is compressed even further.

### Prompt / Prefix Tuning

The pretrained weights remain essentially fixed.

Instead of directly modifying model parameters, these methods modify how the frozen model is **conditioned**.

---

# Computer Vision and LLMs

## 31. The Same Principle Across Different Model Families

In Computer Vision:

```text
ImageNet
   ↓
Pretrained ResNet / EfficientNet
   ↓
Target Dataset
   ↓
Feature Extraction
or
Partial / Full Fine-Tuning
```

In LLMs:

```text
Large-Scale Pre-training
   ↓
Pretrained LLM
   ↓
Target Task
   ↓
Full Fine-Tuning
or
PEFT
```

Despite the architectural differences, the underlying principle is the same:

$$
\boxed{
\text{Do Not Relearn Everything From Scratch}
}
$$

Instead:

$$
\boxed{
\text{Reuse a Powerful Pretrained Representation and Adapt Only What Is Necessary}
}
$$

---

# Practical Decision Framework

## 32. How to Choose an Adaptation Strategy

### 1. How much target data is available?

If the dataset is small:

$$
\text{Freeze More Parameters}
$$

If the dataset is large:

$$
\text{More Aggressive Fine-Tuning Becomes Possible}
$$

---

### 2. How different is the target domain?

If the target domain is similar to the source domain:

$$
\text{Pretrained Features May Already Be Sufficient}
$$

If the target domain is very different:

$$
\text{Stronger Adaptation May Be Required}
$$

---

### 3. How large is the model?

For CNNs with tens of millions of parameters:

$$
\text{Full Fine-Tuning Can Be Practical}
$$

For LLMs with billions of parameters:

$$
\text{PEFT Becomes Much More Attractive}
$$

---

### 4. How much GPU memory is available?

If memory is limited for an LLM:

$$
\boxed{\text{LoRA}}
$$

or especially:

$$
\boxed{\text{QLoRA}}
$$

can be practical choices.

---

# What to Learn Deeply

## 33. Learning Priority

### Understand Deeply

```text
Transfer Learning
Feature Extraction
Linear Probing
Partial Fine-Tuning
Full Fine-Tuning
```

The important questions are:

- What knowledge is being transferred?
- Why can pretrained representations be reused?
- Why freeze some layers?
- Why unfreeze other layers?
- How do dataset size and domain similarity affect the strategy?

---

### Especially Important for Generative AI

```text
PEFT
  ↓
LoRA
  ↓
QLoRA
```

The most important equation is:

$$
\boxed{
\Delta W=BA
}
$$

The goal is to understand **why low-rank adaptation reduces the number of trainable parameters**, not merely memorize that LoRA requires less GPU memory.

---

### Understand the Core Idea

```text
Adapters
Prompt Tuning
Prefix Tuning
LoRA-XS
TinyLoRA
```

Detailed implementations are less important than understanding where they fit conceptually.

---

# Essence

## 34. Final Mental Model

The entire topic can be compressed into one chain:

```text
Pre-training
    ↓
Learn Reusable Knowledge
    ↓
Pretrained Model
    ↓
Transfer Learning
    ↓
Reuse Knowledge for a New Task
    ↓
Choose How Much to Adapt
```

If the pretrained representation is already sufficient:

$$
\boxed{\text{Feature Extraction}}
$$

If the representation needs adaptation:

$$
\boxed{\text{Fine-Tuning}}
$$

Fine-tuning ranges from:

$$
\boxed{
\text{Partial Fine-Tuning}
\rightarrow
\text{Full Fine-Tuning}
}
$$

When the model becomes too large for convenient full fine-tuning:

$$
\boxed{\text{PEFT}}
$$

becomes attractive.

The central idea of LoRA is:

$$
W'=W_0+\Delta W
$$

but instead of learning a full update:

$$
\Delta W
$$

LoRA learns:

$$
\boxed{
\Delta W=BA
}
$$

QLoRA extends this idea:

$$
\boxed{
\text{QLoRA}
=
\text{4-bit Frozen Base Model}
+
\text{LoRA}
}
$$

At the deepest level:

$$
\boxed{
\text{Pre-training learns capabilities}
}
$$

$$
\boxed{
\text{Transfer Learning reuses capabilities}
}
$$

$$
\boxed{
\text{Fine-Tuning specializes capabilities}
}
$$

$$
\boxed{
\text{PEFT specializes them using very few trainable parameters}
}
$$

# Data Augmentation

## 1. Core Idea

Data Augmentation is a technique that increases the diversity of training data by applying transformations to existing samples.

Given a training example:

$$
(x,y)
$$

we apply a transformation:

$$
T(x)
$$

while preserving the semantic target:

$$
y(T(x))=y(x)
$$

Thus:

$$
\boxed{
(x,y)
\rightarrow
(T(x),y)
}
$$

The deeper idea is:

> **Data Augmentation does not merely create more images. It teaches the model which variations of the input should not change the semantic prediction.**

---

## 2. Why Data Augmentation Is Needed

Deep neural networks can contain millions or billions of parameters.

When the training dataset is small, the model may memorize accidental properties of the training data instead of learning generalizable patterns.

For example, a model may accidentally learn:

```text
Cats usually appear near the center.

Dogs usually face to the right.

Training images are usually bright.

Objects usually occupy a specific image size.
```

Such correlations may work on the training set but fail in real-world conditions.

Data Augmentation exposes the model to more variation:

```text
Original Image
      ↓
Different Crops
Different Positions
Different Lighting
Different Orientations
Different Scales
```

The model is therefore encouraged to learn the underlying concept instead of exact pixel configurations.

---

## 3. Data Augmentation Does Not Create Truly New Information

Suppose we have one original image:

```text
1 Original Image
```

and generate:

```text
100 Cropped / Flipped / Rotated Versions
```

We technically obtain more training samples, but we do not obtain the same amount of information as collecting 100 independent real-world images.

Therefore:

$$
\boxed{
\text{Data Augmentation increases training variation}
}
$$

but does not fully replace:

$$
\boxed{
\text{Collecting genuinely new data}
}
$$

A better interpretation is:

> Data Augmentation expands the neighborhood around the observations we already have.

---

## 4. Learning Invariance

Suppose:

$$
f(x)
$$

is the prediction of the model.

If a transformation $T$ should not change the semantic meaning of the input, then ideally:

$$
\boxed{
f(T(x))\approx f(x)
}
$$

For example:

```text
Cat Facing Left
      ↓
Horizontal Flip
      ↓
Cat Facing Right
```

The class should remain:

$$
\text{Cat}
$$

Therefore, Data Augmentation teaches the model to become **invariant** to transformations that are irrelevant to the task.

Examples:

$$
\text{small translation}
$$

$$
\text{small rotation}
$$

$$
\text{lighting changes}
$$

$$
\text{moderate scale changes}
$$

may leave the semantic identity unchanged.

---

## 5. Mathematical View

Consider a dataset:

$$
D=
\{(x_i,y_i)\}_{i=1}^{N}
$$

Without augmentation, training minimizes the empirical loss:

$$
\frac{1}{N}
\sum_{i=1}^{N}
L(f_\theta(x_i),y_i)
$$

With Data Augmentation, we sample a transformation:

$$
T\sim p(T)
$$

and optimize:

$$
\boxed{
\frac{1}{N}
\sum_{i=1}^{N}
\mathbb{E}_{T\sim p(T)}
\left[
L(f_\theta(T(x_i)),y_i)
\right]
}
$$

Instead of asking:

> Can the model correctly classify this exact training image?

we are asking:

> Can the model correctly classify an entire family of valid transformations of this image?

Conceptually:

$$
x_i
$$

becomes:

$$
\boxed{
\{T(x_i):T\sim p(T)\}
}
$$

---

## 6. Data Augmentation as Regularization

Data Augmentation can also be viewed as a regularization technique.

Its goal is to reduce overfitting and improve generalization.

Different regularization methods act at different places:

```text
L2 Regularization
→ constrain parameters

Dropout
→ perturb internal activations

Data Augmentation
→ perturb training inputs
```

All of them discourage the model from relying too heavily on highly specific training patterns.

Therefore:

$$
\boxed{
\text{Data Augmentation}
=
\text{Regularization in Input Space}
}
$$

---

# Common Data Augmentation Techniques

## 7. Geometric Transformations

Geometric augmentations modify the spatial structure of an image.

Typical examples include:

```text
Horizontal Flip
Random Crop
Translation
Rotation
Scaling / Zoom
Shearing
```

### Horizontal Flip

For object classification:

```text
Dog Facing Right
        ↓
Horizontal Flip
        ↓
Dog Facing Left
```

The object class remains unchanged.

Thus:

$$
y(T(x))=y(x)
$$

---

### Random Crop

Instead of always showing the complete image, we randomly crop part of it and resize the result.

This forces the model to become less dependent on:

```text
exact object position
exact object scale
exact image framing
```

and improves robustness to translation and scale variation.

---

### Rotation

Small random rotations can make the model more robust to changes in camera orientation.

For example:

$$
-10^\circ
\leq
\theta
\leq
10^\circ
$$

may be reasonable for many natural-image tasks.

However, the valid amount of rotation depends entirely on the semantics of the task.

---

## 8. Photometric / Color Augmentation

These transformations change image appearance without significantly changing its geometry.

Examples include:

```text
Brightness
Contrast
Saturation
Hue
Color Jitter
```

The same object may appear under:

```text
Different Lighting
Different Cameras
Different White Balance
Different Weather Conditions
```

but should still be recognized as the same semantic object.

The model should learn:

$$
\boxed{
\text{Object Identity}
\neq
\text{Exact RGB Values}
}
$$

---

# Online and Offline Augmentation

## 9. Offline Augmentation

Augmented images are generated beforehand and stored on disk.

For example:

```text
cat.jpg

cat_flip.jpg
cat_crop.jpg
cat_rotate.jpg
```

Advantages:

- simple to inspect,
- deterministic.

Disadvantages:

- additional storage,
- limited number of generated variants.

---

## 10. Online Augmentation

More commonly, transformations are sampled dynamically during training.

```text
Original Image
      ↓
Random Transformation
      ↓
Training Batch
```

The same image may appear differently across epochs:

```text
Epoch 1
→ Flip + Crop

Epoch 2
→ Different Crop

Epoch 3
→ Color Jitter

Epoch 4
→ No Flip + Different Scale
```

Mathematically:

$$
T\sim p(T)
$$

is sampled repeatedly during training.

This allows the model to see many different variants without storing them explicitly.

---

# Training vs Validation

## 11. Augmentation Is Primarily Applied to the Training Set

A typical pipeline is:

```text
Training Image
      ↓
Random Augmentation
      ↓
Model
```

while validation and test data generally use deterministic preprocessing:

```text
Validation Image
      ↓
Resize / Center Crop
      ↓
Model
```

For example:

```text
Training
---------
RandomResizedCrop
RandomHorizontalFlip
ColorJitter

Validation
----------
Resize
CenterCrop
```

The validation/test set is intended to estimate model performance on the target data distribution, so random training augmentation is usually not applied in the same way.

---

# Semantic Validity

## 12. The Most Important Constraint

An augmentation is valid only when it preserves the semantics required by the task.

We assume:

$$
\boxed{
y(T(x))=y(x)
}
$$

If this assumption is false, augmentation introduces incorrect labels.

Therefore:

$$
\boxed{
\text{There is no universally correct augmentation}
}
$$

The transformation must be chosen according to the task.

---

## 13. Example: Horizontal Flip Is Not Always Valid

For dog classification:

```text
Dog Facing Left
↔
Dog Facing Right
```

the label remains:

$$
\text{Dog}
$$

Therefore horizontal flip is reasonable.

But consider:

```text
Left Arrow
vs
Right Arrow
```

Flipping:

```text
→
```

produces:

```text
←
```

The semantic class changes.

Keeping the original label would create incorrect training data.

Thus augmentation must encode **valid task-specific invariances**.

---

# Classification, Detection, and Segmentation

## 14. Image Classification

For classification:

```text
Image
+
Class Label
```

a transformation such as horizontal flip may preserve the label directly.

For example:

$$
\text{Dog}
\rightarrow
\text{Dog}
$$

---

## 15. Object Detection

Object Detection contains:

```text
Image
+
Bounding Boxes
```

If the image is transformed:

$$
x\rightarrow T(x)
$$

the bounding boxes must also be transformed:

$$
b\rightarrow T(b)
$$

For example, horizontally flipping the image requires horizontally flipping the bounding-box coordinates.

---

## 16. Semantic Segmentation

Semantic Segmentation contains:

```text
Image
+
Pixel-Level Mask
```

If the image is transformed, the segmentation mask must receive exactly the same spatial transformation:

```text
Image → Rotate
Mask  → Rotate
```

Therefore:

$$
\boxed{
\text{Spatial augmentation must transform both input and spatial labels consistently}
}
$$

---

# Augmentation Strength

## 17. Weak vs Strong Augmentation

Augmentations can vary in strength.

### Weak Augmentation

Examples:

```text
Small Crop
Small Rotation
Horizontal Flip
Mild Color Jitter
```

The resulting image remains relatively close to the original.

### Strong Augmentation

Examples may include:

```text
Large Crops
Heavy Color Distortion
Large Geometric Transformations
MixUp
CutMix
```

Strong augmentation increases training diversity but also increases the risk of destroying semantic information.

Therefore there is a trade-off:

$$
\boxed{
\text{More Diversity}
\quad\leftrightarrow\quad
\text{Preserve Semantic Validity}
}
$$

---

# MixUp and CutMix

## 18. MixUp

Traditional augmentation transforms one image.

MixUp combines two training samples:

$$
\tilde{x}
=
\lambda x_i+(1-\lambda)x_j
$$

and mixes their labels:

$$
\tilde{y}
=
\lambda y_i+(1-\lambda)y_j
$$

For example:

```text
70% Cat Image
+
30% Dog Image
```

produces a target approximately corresponding to:

```text
70% Cat
30% Dog
```

The deeper idea is that the model is encouraged to behave smoothly between training samples.

---

## 19. CutMix

CutMix combines two images spatially.

Conceptually:

```text
┌───────────────────┐
│                   │
│      Image A      │
│        ┌─────┐    │
│        │  B  │    │
│        └─────┘    │
│                   │
└───────────────────┘
```

A region from Image B is inserted into Image A.

The target labels are mixed according to the corresponding image areas.

Both MixUp and CutMix can be understood as stronger augmentation and regularization techniques.

---

# Relationship with Dataset Size

## 20. Why Data Augmentation Is Especially Useful for Small Datasets

When:

$$
N_{\text{train}}
$$

is small, the model sees limited natural variation.

Data Augmentation artificially expands that variation.

Therefore:

$$
\boxed{
N_{\text{train}}\downarrow
\quad\Rightarrow\quad
\text{Augmentation becomes especially valuable}
}
$$

However, even very large datasets cannot contain every possible:

```text
pose
lighting condition
camera angle
scale
background
object position
```

so augmentation remains useful even in large-scale training.

---

# Relationship with Transfer Learning

## 21. Data Augmentation + Transfer Learning

Transfer Learning and Data Augmentation solve different but complementary problems.

Transfer Learning answers:

> Where should useful initial representations come from?

Data Augmentation answers:

> How can the target dataset expose the model to more valid variation?

A common workflow is:

```text
Pretrained Model
      ↓
Transfer Learning
      ↓
Freeze / Fine-Tune
      +
Data Augmentation
      ↓
Target Model
```

This combination is particularly useful when the target dataset is small.

---

# Distribution Perspective

## 22. Data Augmentation Expands the Training Distribution

The observed dataset represents only a limited empirical distribution:

$$
p_{\text{data}}(x,y)
$$

Data Augmentation samples transformations:

$$
T\sim p(T)
$$

to produce a broader effective training distribution:

$$
p_{\text{aug}}(x,y)
$$

Conceptually:

```text
Observed Training Distribution
            ↓
       Augmentation
            ↓
Broader Training Distribution
            ↓
More Realistic Variation
```

Therefore Data Augmentation is not simply about increasing the number of examples.

It is about defining:

> **Which variations of the real world should the model learn to tolerate?**

---

# Deeper Interpretation

## 23. Data Augmentation Injects Human Prior Knowledge

Humans already know that many transformations should not change an object's identity.

For example:

```text
Cat Facing Left
and
Cat Facing Right
```

are both cats.

A neural network does not automatically know this.

At the tensor level, the two inputs may be very different.

Data Augmentation injects this prior knowledge by enforcing:

$$
x
\rightarrow
T(x)
$$

while preserving:

$$
y
$$

Therefore:

$$
\boxed{
\text{Data Augmentation}
=
\text{Inject Prior Knowledge About Valid Invariances}
}
$$

This is one of the deepest ways to understand why augmentation works.

---

# Essence

Data Augmentation should not be remembered merely as:

```text
Flip
Crop
Rotate
```

The essential operation is:

$$
\boxed{
(x,y)
\rightarrow
(T(x),y)
}
$$

provided that:

$$
\boxed{
T(x)
\text{ preserves the task-relevant semantics}
}
$$

The desired behavior is:

$$
\boxed{
f(T(x))
\approx
f(x)
}
$$

for transformations to which the prediction should be invariant.

Therefore:

$$
\boxed{
\text{Data Augmentation}
=
\text{Expand Valid Training Variation}
}
$$

$$
\boxed{
=
\text{Inject Invariance Assumptions}
}
$$

$$
\boxed{
=
\text{Regularize the Model in Input Space}
}
$$

$$
\boxed{
=
\text{Improve Generalization}
}
$$

The most important mental model is:

> **Data Augmentation does not simply create more images; it teaches the model which changes in the input should not change the semantic prediction.**

# State of Computer Vision

## 1. Core Idea

The main idea behind the **State of Computer Vision** is that the performance of a learning system depends on two major sources of knowledge:

$$
\boxed{
\text{Knowledge from Data}
+
\text{Knowledge from Human Engineering}
}
$$

When labeled data is limited, we usually need to inject more human knowledge through:

```text
Architecture Design
Transfer Learning
Data Augmentation
Specialized Components
Training Strategies
```

As the amount of representative data increases, the model can learn more useful structure directly from data.

A useful mental model is:

$$
\boxed{
\text{More Data}
\Rightarrow
\text{Less Dependence on Task-Specific Hand-Engineering}
}
$$

This does **not** mean architecture becomes unimportant.

It means that with more data, we can rely more on learning and less on manually encoding task-specific assumptions.

---

## 2. Data vs. Hand-Engineering

Consider a learning problem:

$$
f:X\rightarrow Y
$$

If only a small amount of labeled data is available, many different functions may fit the training set.

The data alone may not provide enough information to determine which solution will generalize well.

Therefore, humans inject additional prior knowledge.

```text
Small Dataset
      ↓
Insufficient Evidence from Data
      ↓
Need Stronger Human Priors
      ↓
Architecture / Transfer Learning / Augmentation
```

When much more data becomes available:

```text
Large Dataset
      ↓
More Evidence About the True Problem Structure
      ↓
Model Can Learn More Directly from Data
      ↓
Less Task-Specific Engineering Required
```

Thus:

$$
\boxed{
\text{Less Data}
\Rightarrow
\text{More Important Inductive Bias}
}
$$

---

# Human Knowledge in Deep Learning

## 3. What Does Hand-Engineering Mean?

Hand-engineering is not limited to traditional handcrafted features such as:

```text
SIFT
HOG
Manually Designed Edge Detectors
```

In deep learning, human engineering also includes:

```text
Network Architecture
Specialized Layers
Loss Functions
Hyperparameters
Training Procedures
Data Processing
```

For example:

```text
ResNet
→ Skip Connections

Inception
→ Parallel Multi-Scale Processing

MobileNet
→ Depthwise Separable Convolution

EfficientNet
→ Compound Scaling
```

The network still learns its weights from data, but humans determine the structure in which learning takes place.

Therefore, architecture design is a way of injecting **inductive bias**.

---

## 4. Two Sources of Knowledge

A trained model receives knowledge from two broad sources.

### Source 1 — Data

Training examples:

$$
(x,y)
$$

allow gradient descent to learn patterns directly from observations.

```text
Training Data
     ↓
Optimization
     ↓
Learned Representations
```

---

### Source 2 — Human Prior Knowledge

Humans encode assumptions through:

```text
Architecture
Training Strategy
Data Augmentation
Loss Design
Task-Specific Components
```

For example, CNNs encode strong assumptions about images:

```text
Local Connectivity
Weight Sharing
Spatial Structure
Hierarchical Feature Extraction
```

Therefore:

$$
\boxed{
\text{Final Model Knowledge}
\approx
\text{Learned Knowledge}
+
\text{Engineered Prior}
}
$$

---

# Why Computer Vision Required Significant Engineering

## 5. Limited Labeled Data

Historically, many Computer Vision tasks did not have extremely large labeled datasets.

This becomes especially important when labels are expensive to create.

For image classification:

```text
Image
  ↓
"Cat"
```

the annotation is relatively simple.

For Object Detection:

```text
Image
  ↓
Class Label
+
Bounding Box Coordinates
```

annotation is much more expensive.

For Semantic Segmentation:

```text
Image
  ↓
Pixel-Level Labels
```

annotation is even more costly.

Thus, more complex vision tasks often have less labeled data.

Conceptually:

```text
More Expensive Labels
        ↓
Less Labeled Data
        ↓
Greater Need for Human Priors
        ↓
More Specialized Design
```

---

## 6. Why Architecture Case Studies Matter

Architectures such as:

```text
AlexNet
VGG
ResNet
Inception
MobileNet
EfficientNet
```

should not be viewed merely as different layer configurations.

Each architecture introduces a reusable engineering idea.

For example:

```text
ResNet
→ Make very deep networks easier to optimize

Inception
→ Efficiently process multiple spatial scales

MobileNet
→ Reduce computation

EfficientNet
→ Scale depth, width, and resolution systematically
```

These architectures demonstrate how human insight can improve learning when data, optimization, or computation alone is not sufficient.

---

# Transfer Learning

## 7. Reusing Knowledge from Other Datasets

Suppose a target task contains only:

$$
N=2,000
$$

images.

Training from scratch requires those 2,000 images to teach the model:

```text
Edges
Textures
Shapes
Object Parts
High-Level Concepts
Target-Specific Decision Boundary
```

This is difficult.

Transfer Learning changes the problem:

```text
Large Source Dataset
        ↓
Pre-training
        ↓
Pretrained Representation
        ↓
Small Target Dataset
        ↓
Fine-Tuning
```

Now the target dataset does not need to provide all the knowledge.

The model already contains knowledge learned from the source dataset.

Therefore:

$$
\boxed{
\text{Small Target Dataset}
+
\text{Pretrained Knowledge}
}
$$

can be much more effective than:

$$
\boxed{
\text{Small Target Dataset Alone}
}
$$

Transfer Learning can therefore be viewed as:

> **Reusing knowledge learned from a much larger dataset to compensate for limited target data.**

---

# Data Augmentation

## 8. Injecting Transformation Knowledge

Data Augmentation approaches the limited-data problem from another direction.

Given:

$$
(x,y)
$$

we apply a transformation:

$$
T(x)
$$

while preserving the semantic target:

$$
y(T(x))=y(x)
$$

Thus:

$$
\boxed{
(x,y)
\rightarrow
(T(x),y)
}
$$

For example:

```text
Cat Facing Left
      ↓
Horizontal Flip
      ↓
Cat Facing Right
```

Humans know that both images still represent a cat.

Data Augmentation injects this knowledge into the training process.

Therefore:

$$
\boxed{
\text{Data Augmentation}
=
\text{Human Prior About Valid Input Transformations}
}
$$

---

# A Unified View

## 9. Different Ways to Compensate for Limited Data

Architecture design, Transfer Learning, and Data Augmentation can be viewed as different ways of providing information that the target dataset alone may not contain.

```text
                    Limited Target Data
                            │
             ┌──────────────┼──────────────┐
             │              │              │
             ▼              ▼              ▼
      Architecture     Transfer       Data
        Design         Learning     Augmentation
             │              │              │
             ▼              ▼              ▼
      Structural       Reuse Learned   Inject Valid
        Prior            Knowledge     Transformations
             │              │              │
             └──────────────┼──────────────┘
                            ▼
                   Better Generalization
```

Thus:

$$
\boxed{
\text{Architecture}
=
\text{Structural Prior}
}
$$

$$
\boxed{
\text{Transfer Learning}
=
\text{Reuse Learned Knowledge}
}
$$

$$
\boxed{
\text{Data Augmentation}
=
\text{Transformation Prior}
}
$$

All three reduce the amount of information that must be learned exclusively from the target dataset.

---

# Benchmark Optimization

## 10. Ensembling

When benchmark accuracy is extremely important, one technique is to combine several independently trained models.

Suppose:

$$
f_1(x),f_2(x),\ldots,f_K(x)
$$

are different models.

Their predictions can be averaged:

$$
\boxed{
\hat{y}
=
\frac{1}{K}
\sum_{k=1}^{K}f_k(x)
}
$$

Conceptually:

```text
                ┌── Model 1 ──┐
                │             │
Image ──────────├── Model 2 ──┼──→ Average → Prediction
                │             │
                ├── Model 3 ──┤
                │             │
                └── Model K ──┘
```

Different models may make different errors.

If their errors are not perfectly correlated, averaging their predictions can reduce variance.

Therefore:

$$
\boxed{
\text{Ensembling}
\approx
\text{Average Across Models}
}
$$

The trade-off is higher:

```text
Inference Cost
Model Storage
Latency
Compute
```

---

## 11. Test-Time Augmentation / Multi-Crop Evaluation

Instead of evaluating one test image only once, we can generate several valid views:

$$
T_1(x),T_2(x),\ldots,T_K(x)
$$

and average their predictions:

$$
\boxed{
\hat{y}
=
\frac{1}{K}
\sum_{k=1}^{K}
f(T_k(x))
}
$$

For example:

```text
Original Image
      │
      ├── Center Crop
      ├── Top-Left Crop
      ├── Top-Right Crop
      ├── Bottom-Left Crop
      ├── Bottom-Right Crop
      └── Flipped Versions
                ↓
          Multiple Predictions
                ↓
              Average
```

This idea is often referred to more generally as **Test-Time Augmentation (TTA)**.

---

## 12. Training Augmentation vs. Test-Time Augmentation

### Training-Time Data Augmentation

```text
Training Image
      ↓
Random Transformation
      ↓
Model Training
```

Its goal is:

$$
\boxed{
\text{Improve the Learned Model}
}
$$

It changes the learned parameters:

$$
\theta
$$

---

### Test-Time Augmentation

```text
Test Image
    ↓
Multiple Transformations
    ↓
Multiple Predictions
    ↓
Average
```

Its goal is:

$$
\boxed{
\text{Improve the Final Prediction}
}
$$

It does not train the model further.

---

## 13. Ensemble vs. Test-Time Augmentation

These techniques are conceptually similar.

### Ensembling

$$
\boxed{
\text{Average Across Models}
}
$$

```text
Same Input
   ↓
Multiple Models
   ↓
Average
```

### Test-Time Augmentation

$$
\boxed{
\text{Average Across Views}
}
$$

```text
Multiple Views
   ↓
Same Model
   ↓
Average
```

Both methods reduce sensitivity to a particular model or input configuration.

---

# Benchmark vs. Production

## 14. Best Accuracy Is Not Always the Best System

Suppose:

```text
Single Model

Accuracy = 94.5%
Compute  = 1×
```

Using several models and several test-time crops might produce:

```text
Ensemble + TTA

Accuracy = 95.2%
Compute  = 30×
```

For a benchmark competition, the extra accuracy may be worthwhile.

For a mobile application, the extra compute may be unacceptable.

Therefore:

$$
\boxed{
\text{Best Benchmark Model}
\neq
\text{Best Production System}
}
$$

Model evaluation should therefore consider not only:

$$
\text{Accuracy}
$$

but also:

$$
\text{Latency}
$$

$$
\text{Memory}
$$

$$
\text{Compute}
$$

$$
\text{Energy Consumption}
$$

---

# Reusing Existing Work

## 15. Open-Source Implementations and Pretrained Models

Modern Computer Vision systems rarely need to recreate every successful architecture from scratch.

A practical workflow is:

```text
Published Architecture
        ↓
Trusted Implementation
        ↓
Pretrained Weights
        ↓
Target Dataset
        ↓
Fine-Tuning
```

Instead of:

```text
Read Paper
   ↓
Implement Everything
   ↓
Debug Architecture
   ↓
Train from Scratch
```

we can reuse established engineering and focus on the target problem.

This creates another form of knowledge transfer:

$$
\boxed{
\text{Reuse Engineering Knowledge}
+
\text{Reuse Learned Parameters}
}
$$

---

# Data and Architecture

## 16. More Data Does Not Make Architecture Irrelevant

A larger dataset reduces the need for manually engineered task-specific assumptions.

However, architecture still affects:

```text
Optimization
Compute Efficiency
Memory
Sample Efficiency
Inductive Bias
Training Stability
```

Therefore:

$$
\boxed{
\text{More Data}
\not\Rightarrow
\text{Architecture Is Irrelevant}
}
$$

A more accurate statement is:

$$
\boxed{
\text{More Data}
\Rightarrow
\text{Less Dependence on Task-Specific Hand-Engineering}
}
$$

---

# Connecting the Ideas

## 17. The Overall Story

The major ideas can be connected as:

```text
Architecture Case Studies
        ↓
Learn useful structural priors

Open-Source Implementations
        ↓
Reuse engineering knowledge

Transfer Learning
        ↓
Reuse knowledge learned from large datasets

Data Augmentation
        ↓
Inject prior knowledge about valid transformations

State of Computer Vision
        ↓
Understand why these techniques are especially valuable
when labeled target data is limited
```

The central theme is:

> When the model cannot learn enough knowledge from the available target data alone, additional knowledge must come from somewhere else.

That knowledge may come from:

```text
Architecture
Other Datasets
Pretrained Models
Human Priors
Data Augmentation
Existing Implementations
```

---

# Essence

The deepest idea can be summarized as:

$$
\boxed{
\text{Learning System Knowledge}
=
\text{Knowledge from Data}
+
\text{Knowledge from Human Engineering}
}
$$

When:

$$
\text{Data}\downarrow
$$

the importance of:

$$
\text{Architecture}
+
\text{Transfer Learning}
+
\text{Data Augmentation}
+
\text{Human Priors}
$$

increases.

When:

$$
\text{Data}\uparrow
$$

the model can learn more useful structure directly from data.

A practical Computer Vision workflow is therefore:

```text
Use Proven Architecture
        +
Use Trusted Implementation
        +
Use Pretrained Weights
        +
Apply Data Augmentation
        +
Fine-Tune on Target Data
```

If benchmark performance is the main objective:

```text
Ensembling
+
Test-Time Augmentation
```

can further improve accuracy at the cost of additional computation.

The most important mental model is:

> **When target data does not provide enough information by itself, we compensate by injecting knowledge through architecture, pretrained representations, augmentation, and training strategy.**